In [ ]:
!pip install -q "crewai>=0.74,<1.0" "crewai-tools>=0.13,<1.0" \ "google-genai>=1.40,<2.0" "python-dotenv>=1.0" "litellm>=1.50.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.2/473.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.2/39.2 MB 20.6 MB/

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
os.environ["GEMINI_API_KEY"] =
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
os.environ["SERPER_API_KEY"] =

In [ ]:
from crewai import LLM
GOOGLE_KEY = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
assert GOOGLE_KEY, "Set GEMINI_API_KEY or GOOGLE_API_KEY first."

In [ ]:
my_llm = LLM(
    api_key=GOOGLE_KEY,
    model="gemini/gemini-2.0-flash",
)

In [ ]:
from crewai_tools import SerperDevTool
search_tool = SerperDevTool()

In [ ]:
from crewai import Agent, Task, Crew

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic} in 'https://medium.com/'."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "You have to prepare a detailed "
              "outline and the relevant topics and sub-topics that has to be a part of the"
              "blogpost."
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    verbose=True,
    llm=my_llm,
    tools=[search_tool]
)

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate blog posts based on the planner's outline.",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic} in 'https://medium.com/'. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True,
    llm=my_llm,
)

In [ ]:
editor = Agent(
    role="Editor",
    goal="Refine the blog post for readability, grammar, and tone.",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True,
    llm=my_llm,
)

In [ ]:
plan_task = Task(
    description="Research and outline a full content plan for {topic} with headings, subtopics, SEO keywords, and audience focus.",
    expected_output="Structured outline with sections, keywords, and content direction.",
    agent=planner,
)

write_task = Task(
    description="Using the planner's outline, write a detailed, engaging blog post with 2–3 paragraphs per section.",
    expected_output="Complete blog post in markdown, suitable for Medium.",
    agent=writer,
)

edit_task = Task(
    description="Proofread and refine the blog post for clarity, flow, and grammar.",
    expected_output="Final polished blog post.",
    agent=editor,
)

Or be more specific


In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner
)


write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
  "3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,

)




edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."
                 ),
    expected_output="A well-written blog post in markdown format, "
                    "without the word markdown in the beginning "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor,
)

In [ ]:
agents = [planner, writer, editor]
crew = Crew(agents=agents, tasks=[plan_task, write_task, edit_task], verbose=True)

In [ ]:
result = crew.kickoff(inputs={"topic": "Agentic AI"})
print(result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d4a8d96a-9ee4-4528-adbd-53f2649a6666                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: Research and outline a full content plan for Agentic AI with headings, subtopics, SEO keywords, and      │
│  audience focus.                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Thought: Okay, I understand. I need to research and create a detailed outline for a blog post about Agentic    │
│  AI on Medium. The outline should include sections, subtopics, SEO keywords, and a clear content direction.     │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "Agentic AI definition, benefits, challenges, examples"                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Agentic AI: The Next Evolution of Artificial Intelligence                                                    │
│                                                                                                                 │
│  Imagine a world where AI doesn't just respond to commands, but proactively identifies and solves problems,     │
│  making decisions with minimal human input. That future is closer than you think, driven by Agentic AI.         │
│                                                                                                                 │
│  Agentic AI refers to artificial intelligence systems engineered to independently set goals, strategize, make   │
│  decisions, and execute actions to achieve those goals with minimal human intervention. These systems are       │
│  designed as autonomous problem-solvers, heralding a new era of AI capabilities.                                │
│                                                                                                                 │
│  Agentic AI distinguishes itself from traditional AI and Generative AI in significant ways. Traditional AI      │
│  operates through rule-based systems, executing specific tasks based on pre-programmed instructions with        │
│  limited autonomy or adaptability. Generative AI excels at creating new content, such as text and images, but   │
│  primarily generates outputs rather than acting independently to achieve specific goals. Agentic AI, however,   │
│  goes beyond both by exhibiting autonomy, planning, and decision-making capabilities to achieve specific        │
│  objectives.                                                                                                    │
│                                                                                                                 │
│  This article will explore the potential benefits, challenges, and real-world applications of Agentic AI.       │
│  Representing a significant leap forward, Agentic AI promises to revolutionize industries and reshape how we    │
│  interact with technology. This article dives deep into this exciting field, providing a comprehensive          │
│  overview for anyone eager to understand its transformative potential.                                          │
│                                                                                                                 │
│  # What is Agentic AI?                                                                                          │
│                                                                                                                 │
│  Agentic AI is characterized by four key features: autonomy, reasoning, planning, and action. Autonomy allows   │
│  these systems to operate independently, reducing the need for constant human oversight. Reasoning enables      │
│  them to analyze information and make logical decisions. Planning involves developing strategies and sequences  │
│  of actions to achieve desired outcomes, and action refers to the ability to execute those plans and interact   │
│  with the environment.                                                                                          │
│                                                        

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 9224523b-38cf-4b54-93c9-54d7ba710a24                                                                     │
│  Agent: Editor                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d4a8d96a-9ee4-4528-adbd-53f2649a6666                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ```markdown                                                                                      │
│  # Agentic AI: The Next Evolution of Artificial Intelligence                                                    │
│                                                                                                                 │
│  Imagine a world where AI doesn't just respond to commands, but proactively identifies and solves problems,     │
│  making decisions with minimal human input. That future is closer than you think, driven by Agentic AI.         │
│                                                                                                                 │
│  Agentic AI refers to artificial intelligence systems engineered to independently set goals, strategize, make   │
│  decisions, and execute actions to achieve those goals with minimal human intervention. These systems are       │
│  designed as autonomous problem-solvers, heralding a new era of AI capabilities.                                │
│                                                                                                                 │
│  Agentic AI distinguishes itself from traditional AI and Generative AI in significant ways. Traditional AI      │
│  operates through rule-based systems, executing specific tasks based on pre-programmed instructions with        │
│  limited autonomy or adaptability. Generative AI excels at creating new content, such as text and images, but   │
│  primarily generates outputs rather than acting independently to achieve specific goals. Agentic AI, however,   │
│  goes beyond both by exhibiting autonomy, planning, and decision-making capabilities to achieve specific        │
│  objectives.                                                                                                    │
│                                                                                                                 │
│  This article will explore the potential benefits, challenges, and real-world applications of Agentic AI.       │
│  Representing a significant leap forward, Agentic AI promises to revolutionize industries and reshape how we    │
│  interact with technology. This article dives deep into this exciting field, providing a comprehensive          │
│  overview for anyone eager to understand its transformative potential.                                          │
│                                                                                                                 │
│  # What is Agentic AI?                                                                                          │
│                                                                                                                 │
│  Agentic AI is characterized by four key features: autonomy, reasoning, planning, and action. Autonomy allows   │
│  these systems to operate independently, reducing the need for constant human oversight. Reasoning enables      │
│  them to analyze information and make logical decisions. Planning involves developing strategies and sequences  │
│  of actions to achieve desired outcomes, and action refers to the ability to execute those plans and interact   │
│  with the environment.                                

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): n
```markdown
# Agentic AI: The Next Evolution of Artificial Intelligence

Imagine a world where AI doesn't just respond to commands, but proactively identifies and solves problems, making decisions with minimal human input. That future is closer than you think, driven by Agentic AI.

Agentic AI refers to artificial intelligence systems engineered to independently set goals, strategize, make decisions, and execute actions to achieve those goals with minimal human intervention. These systems are designed as autonomous problem-solvers, heralding a new era of AI capabilities.

Agentic AI distinguishes itself from traditional AI and Generative AI in significant ways. Traditional AI operates through rule-based systems, executing specific tasks based on pre-programmed instructions with limited autonomy or adaptability. Generative AI excels at creating new content, such as text and images, but primarily generates outputs rathe